# 2. Prompt Management with DataRobot

*Estimated time to run notebook: about 12-15 min*

**Goal:** Create, version, and programmatically retrieve a DataRobot Prompt Template so your agent can load a consistent system prompt at runtime.

**Flow:** Run the helper cell, then the main cell: the DataRobot Python client (`PromptTemplate`) creates the clinic template when `PROMPT_TEMPLATE_ID` is unset (and prints the ID for `.env`), or loads your template when that variable is set.

**Key concept:** Prompts are managed, versioned platform assets (not hardcoded strings). After setup, you pick a version, render variables, and pass the rendered text as the agent’s `system_prompt`.

If you create or edit prompt templates manually in the DataRobot UI instead of this notebook, follow [Create and manage prompts](https://docs.datarobot.com/latest/en/docs/agentic-ai/prompt-mgmt/create-prompts.html) and the Python client reference for [prompt templates](https://docs.datarobot.com/en/docs/api/reference/sdk/gen-prompting.html).

## What the notebook does

1. Run the helper cell, then the main cell: create a template when `PROMPT_TEMPLATE_ID` is unset or load it from `.env` when set. After a successful run, follow **Save `PROMPT_TEMPLATE_ID` for notebook 4** at the end of this notebook if the helper printed a new ID.
2. Render the prompt with variables (e.g., `company_name`).
3. Use the rendered text as the agent’s `system_prompt` for a quick runtime validation.


In [ ]:
from datarobot.models.genai.prompt_template import PromptTemplate, Variable


DEFAULT_CLINIC_PROMPT_NAME = "System Prompt - Notebooks"
DEFAULT_CLINIC_PROMPT_DESCRIPTION = "Used for Agent Build Clinic"
DEFAULT_CLINIC_PROMPT_TEXT = """You are a helpful forecasting assistant working for {{ company_name }}.

- Use the forecasting deployment with ID {{ forecast_deployment }}.
- Forecast using the scoring dataset with ID {{ scoring_dataset }}.
"""


def default_clinic_variables() -> list[Variable]:
    return [
        Variable(name="company_name", type="str", description="Name of the company"),
        Variable(
            name="forecast_deployment",
            type="str",
            description="ID of the forecast deployment",
        ),
        Variable(name="scoring_dataset", type="str", description="ID of the scoring dataset"),
    ]


def print_prompt_template_access(template: PromptTemplate) -> None:
    """Print template ID and a ready-to-paste ``.env`` line."""
    print("\n" + "=" * 60)
    print("  PROMPT TEMPLATE — add to repo root .env for notebooks 2 and 4")
    print("=" * 60)
    print(f"  PROMPT_TEMPLATE_ID={template.id}")
    print(f"  Name: {template.name}")
    desc = getattr(template, "description", "") or "(none)"
    print(f"  Description: {desc}")
    print("=" * 60 + "\n")


def setup_retrieve_prompt_template(
    prompt_template_id: str | None,
    *,
    create_if_missing: bool = True,
    name: str = DEFAULT_CLINIC_PROMPT_NAME,
    description: str = DEFAULT_CLINIC_PROMPT_DESCRIPTION,
    prompt_text: str = DEFAULT_CLINIC_PROMPT_TEXT,
    variables: list[Variable] | None = None,
    commit_comment: str = "Initial version (Agent Build Clinic notebook)",
) -> PromptTemplate:
    """Create a new prompt template (with v1) or fetch by ID; always print ID for user access.

    If ``prompt_template_id`` is set, loads that template from DataRobot.
    If unset and ``create_if_missing`` is True, creates the template and first version
    using the clinic defaults defined in this notebook.
    """
    if prompt_template_id and prompt_template_id.strip():
        template = PromptTemplate.get(prompt_template_id.strip())
        print(f"Retrieved prompt template: {template.id}")
        print_prompt_template_access(template)
        return template

    if not create_if_missing:
        raise ValueError(
            "PROMPT_TEMPLATE_ID is not set and create_if_missing=False. "
            "Add PROMPT_TEMPLATE_ID to .env or pass create_if_missing=True."
        )

    template = PromptTemplate.create(name=name, description=description)
    vars_ = variables if variables is not None else default_clinic_variables()
    template.create_version(
        prompt_text=prompt_text,
        variables=vars_,
        commit_comment=commit_comment,
    )
    print("Created prompt template and initial version in DataRobot.")
    print_prompt_template_access(template)
    return template


In [ ]:
# 1. Imports & setup
import os
from dotenv import load_dotenv
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# 2. Load configuration
load_dotenv(override=True)

# 3. Initialize DataRobot client
dr_client = dr.Client()
print(f"Connected to DataRobot: {dr_client.endpoint}")

# 4. Read configuration from environment
FORECAST_DEPLOYMENT_ID = os.getenv("FORECAST_DEPLOYMENT_ID")
SCORING_DATASET_ID = os.getenv("SCORING_DATASET_ID")
# Version to test (set to "v1", "v2", or None for latest)
PROMPT_VERSION_ID = "v1"

RENDER_VARIABLES = {
    "company_name": "DataRobot Forecasting Inc.",
    "forecast_deployment": FORECAST_DEPLOYMENT_ID,
    "scoring_dataset": SCORING_DATASET_ID,
}

MODEL_NAME = os.getenv("MODEL_NAME")

# 5. Setup or retrieve prompt template (run previous cell for ``setup_retrieve_prompt_template``)
_raw_prompt_id = os.getenv("PROMPT_TEMPLATE_ID")
template = setup_retrieve_prompt_template(
    _raw_prompt_id.strip() if _raw_prompt_id else None,
    create_if_missing=True,
)
PROMPT_TEMPLATE_ID = template.id

print(f"\n--- Using PROMPT TEMPLATE ID: {PROMPT_TEMPLATE_ID} ---")
print(f"Template Name: {template.name}")

target_version = None
if PROMPT_VERSION_ID:
    # Logic to handle "v1", "V1", or 1
    versions = template.list_versions()
    search_str = str(PROMPT_VERSION_ID).lower().replace("v", "")

    for v in versions:
        # Check exact ID match OR version number match
        if v.id == PROMPT_VERSION_ID:
            target_version = v
            break
        if hasattr(v, "version") and str(v.version) == search_str:
            target_version = v
            break

    if not target_version:
        available = [f"v{getattr(v, 'version', '?')} (ID: {v.id})" for v in versions]
        raise ValueError(
            f"Could not find version '{PROMPT_VERSION_ID}'.\nAvailable: {available}"
        )
else:
    print("Fetching latest version...")
    target_version = template.get_latest_version()

print(
    f"Selected Version: v{getattr(target_version, 'version', '?')} (ID: {target_version.id})"
)

# 6. Render prompt text (this becomes the agent's system_prompt)
try:
    system_prompt = target_version.render(variables=RENDER_VARIABLES)
    print("\n" + "=" * 30)
    print("  RENDERED SYSTEM PROMPT")
    print("=" * 30)
    print(system_prompt)
    print("=" * 30)
except Exception as e:
    print(f"\nCRITICAL ERROR: {e}")
    if hasattr(target_version, "variables"):
        print(f"REQUIRED Variables: {target_version.variables}")
    print(f"PROVIDED Variables: {list(RENDER_VARIABLES.keys())}")
    raise e

# 7. Configure the LLM (via DataRobot LLM Gateway)
llm = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

# 8. Define the agent using the rendered prompt
agent = Agent(model=llm, system_prompt=system_prompt)

# 9. Execution (sanity-check)
print("\n--- Running Test Query ---")
test_query = "Hello, who are you and who do you work for?"
print(f"User: {test_query}\n")

async with agent:
    response = await agent.run(test_query)
    print(response.output)

#### Save `PROMPT_TEMPLATE_ID` for notebook 4

After the main cell runs, check the printed block titled **PROMPT TEMPLATE ID** (from the helper).

1. Open the **`.env`** file at the **repository root** (same folder as the notebooks). 
2. Set **`PROMPT_TEMPLATE_ID`** to that variable (uncomment the line if it is commented). Save the file.

**4 - Forecast Agent Tools.ipynb** reads `PROMPT_TEMPLATE_ID` from `.env` so it uses the same prompt template as this notebook. If you already had `PROMPT_TEMPLATE_ID` set before running the main cell, the printed ID matches what is in `.env` and you do not need to change it.